In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import pandas as pd
import subprocess, os
from sklearn.decomposition import PCA
import anndata as ad
import glob

In [ ]:
def run_tissue_mosaic(dsetname, filename_parser):
    os.makedirs(f'_embeddings', exist_ok=True)

    subprocess.run(["python", "2.tissuemosaic.sh", dsetname])

    def preprocess_featurized_h5ad(file): 
        d = sc.read_h5ad(file) 
        d = d[d.obs.dino_spot_features_valid == True]
        return(d) 
    
    d_full = ad.concat([preprocess_featurized_h5ad(file) for file in glob.glob(f'_data/{dsetname}/sample_ads_featurized/*')])
    embedding = d_full.obsm['dino_spot_features']
    print(embedding.shape)
    sample_names = d_full.obs.sid
    unique_samples = np.unique(sample_names)
    print(f'Loaded {len(sample_names)} samples, {len(unique_samples)} unique samples')
    obs = pd.DataFrame(sample_names, columns=['fullname'])
    obs.fullname = obs.fullname.str.replace('-', '.')
    obs['sid'] = obs.fullname.apply(lambda x: filename_parser(x)['sid'])
    obs['donor'] = obs.fullname.apply(lambda x: filename_parser(x)['donor'])
    obs['method_cluster'] = d_full.obs.leiden1
    
    d = sc.AnnData(X=embedding,
              obs=obs)
    sc.pp.neighbors(d)
    sc.tl.umap(d)
    sc.tl.leiden(d, resolution=1)
    d.write(f'_embeddings/{dsetname}_tissuemosaic_noharm.h5ad')